# Chapter 8 — Task Decomposition Strategies · Practice

**Goal:** build every decomposition shape from the chapter by hand (a gated chain, an adaptive planner loop, per-file plus integration review, an independent reviewer, a calibrated verification pass, routing, parallelization, evaluator-optimizer and a bounded async fan-out) and finish with a research orchestrator that decomposes, fans out, synthesizes and gets independently reviewed.

- Theory: [README.md](README.md) (read section 8.x before running the matching section here)
- Estimated time: 3–4 hours
- Estimated API cost: **about $0.30–0.60** with the default model (`claude-sonnet-5-5`) if you run every live cell once, mostly from the multi-pass review and the mini-project. Every section also has **offline** cells that run with no key and no credit.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 2](../02-tool-use/README.md) (`messages.parse`, structured outputs), basic `asyncio` (`async`/`await`, `asyncio.gather`)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. Uses `FakeClaude`: a **real** `anthropic.Anthropic` / `AsyncAnthropic` client whose HTTP layer is replaced by scripted replies, so the SDK's own retries, `messages.parse()` and response types run for real. Only the model is fake. |
| **(live)** | Calls the real API. Without a key it prints `skipped` and does nothing; no offline cell depends on a live cell. |

Async cells use top-level `await`, which Jupyter supports directly (no `asyncio.run`).

In [ ]:
import asyncio
import hashlib
import json
import os
import re
import time
from collections import Counter
from fnmatch import fnmatch
from itertools import count
from typing import Any, Callable, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

load_dotenv(find_dotenv())                     # ANTHROPIC_API_KEY comes from .env, never hardcode it
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
FAST_MODEL = "claude-haiku-5-5"                # the current small model, made for classification and routing

client = anthropic.Anthropic()                 # sequential steps (only the live cells use it)
aclient = anthropic.AsyncAnthropic()           # fan-out: same key, same retries, awaitable calls


def text_of(response) -> str:
    """Join text blocks only; a response can start with a thinking block."""
    return "".join(b.text for b in response.content if b.type == "text")


def parsed_or_raise(response):
    """Structured outputs guarantee shape only when the model finished normally."""
    if response.stop_reason != "end_turn" or response.parsed_output is None:
        raise RuntimeError(f"step failed: stop_reason={response.stop_reason}")
    return response.parsed_output


print("anthropic", anthropic.__version__, "| MODEL =", MODEL, "| FAST_MODEL =", FAST_MODEL)

In [ ]:
# Live cells need a key. Without one, each (live) cell prints "skipped" and every (offline) cell still runs.
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
print("API key found: live cells will run." if HAS_KEY
      else "No ANTHROPIC_API_KEY: (live) cells will be skipped; every (offline) cell still runs.")

### Offline toolkit (run this once)

`FakeClaude(script, latency=...)` stands in for the API. `script` is either a list of replies served in order, or a **function** that receives the request body and returns a reply; use a function for parallel calls, because their arrival order is not deterministic. Helpers: `reply()`, `reply_json()` (what `messages.parse` expects), `api_error()` (for 400 / 429 / 500 tests), and `system_text()` / `user_text()` to inspect a request.

It also records `.requests`, the peak concurrency `.max_in_flight`, and simulates prompt caching: a prefix ending in `cache_control` only becomes readable once an earlier response with the same prefix has begun, which is exactly the rule in the docs.

`PRICES` and `cost_usd()` turn `usage` into dollars (standard API prices per million tokens, checked November 2026 on the [pricing page](https://platform.claude.com/docs/en/about-claude/pricing)).

In [ ]:
_ids = count(1)


def reply(content: str | list[dict], stop_reason: str = "end_turn") -> dict:
    """A Messages API response body (wire format). `content` is a text string or a list of blocks."""
    blocks = [{"type": "text", "text": content}] if isinstance(content, str) else content
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": blocks, "stop_reason": stop_reason, "stop_sequence": None}


def reply_json(obj: Any, stop_reason: str = "end_turn") -> dict:
    """A reply whose text is JSON, as structured outputs (`messages.parse`) return it."""
    data = obj.model_dump() if isinstance(obj, BaseModel) else obj
    return reply(json.dumps(data), stop_reason=stop_reason)


def api_error(status: int, err_type: str, message: str, headers: dict | None = None) -> dict:
    """A scripted HTTP error, e.g. api_error(429, "rate_limit_error", "...", {"retry-after-ms": "50"})."""
    return {"_status": status, "_headers": headers or {}, "type": "error",
            "error": {"type": err_type, "message": message}}


def _texts(content: Any) -> list[str]:
    if isinstance(content, str):
        return [content]
    return [b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"]


def system_text(body: dict) -> str:
    return "".join(_texts(body.get("system", "")))


def user_text(body: dict) -> str:
    """Text of the LAST user message in a request body."""
    users = [m for m in body["messages"] if m["role"] == "user"]
    return "".join(_texts(users[-1]["content"])) if users else ""


def _tokens(s: str) -> int:
    return len(s) // 4 + 1          # rough estimate; real counts come from the API's `usage`


class FakeClaude:
    """Offline Messages API with real SDK clients on top.

    script: a list of replies served in order, or a function `body -> reply` (use a function for
            concurrent calls, whose arrival order is not deterministic).
    .client / .aclient: real `anthropic.Anthropic` / `AsyncAnthropic` whose HTTP transport is this fake,
            so SDK retries, `messages.parse()` and response types all run for real.
    Records every request body in .requests, every usage in .usages, and the peak number of concurrent requests in .max_in_flight.
    Simulates prompt caching: a prefix ending in `cache_control` is written on first use and can be
    read only by requests that ARRIVE after an earlier response with that prefix has begun.
    """

    def __init__(self, script, latency: float = 0.0, max_retries: int = 2, min_cacheable: int = 512):
        self.script = script if callable(script) else list(script)
        self.latency, self.min_cacheable = latency, min_cacheable
        self.requests: list[dict] = []
        self.usages: list[dict] = []          # usage of every successful response, in completion order
        self.in_flight = self.max_in_flight = 0
        self._cache_ready: set[str] = set()
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=max_retries,
            http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(self._handle_sync)))
        self.aclient = anthropic.AsyncAnthropic(
            api_key="offline", max_retries=max_retries,
            http_client=anthropic.DefaultAsyncHttpxClient(transport=httpx2.MockTransport(self._handle_async)))

    def _next(self, body: dict) -> dict:
        if callable(self.script):
            return dict(self.script(body))
        if not self.script:
            return api_error(500, "api_error", "FakeClaude: script exhausted (did a loop not stop?)")
        return dict(self.script.pop(0))

    def _prefix(self, body: dict) -> str | None:
        """Text of the system prompt up to the last block marked with cache_control (if any)."""
        system = body.get("system")
        if not isinstance(system, list):
            return None
        marked = [i for i, b in enumerate(system) if b.get("cache_control")]
        return "".join(b.get("text", "") for b in system[: marked[-1] + 1]) if marked else None

    def _usage(self, body: dict, out: dict, cache_hit: bool) -> dict:
        total_in = _tokens(json.dumps(body.get("system", "")) + json.dumps(body["messages"]))
        prefix = self._prefix(body)
        cached = _tokens(prefix) if prefix and _tokens(prefix) >= self.min_cacheable else 0
        usage = {"input_tokens": total_in - cached, "output_tokens": _tokens("".join(_texts(out["content"]))),
                 "cache_creation_input_tokens": 0 if cache_hit else cached,
                 "cache_read_input_tokens": cached if cache_hit else 0}
        if cached:
            self._cache_ready.add(prefix)     # the response has begun: the entry is now readable
        self.usages.append(usage)
        return usage

    def _response(self, body: dict, out: dict, cache_hit: bool) -> httpx2.Response:
        status, headers = out.pop("_status", 200), out.pop("_headers", {})
        out.pop("_delay", None)
        if status == 200:
            out.setdefault("usage", self._usage(body, out, cache_hit))
        return httpx2.Response(status, json=out, headers=headers)

    def _handle_sync(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        cache_hit = self._prefix(body) in self._cache_ready
        return self._response(body, self._next(body), cache_hit)

    async def _handle_async(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        cache_hit = self._prefix(body) in self._cache_ready      # decided when the request ARRIVES
        self.in_flight += 1
        self.max_in_flight = max(self.max_in_flight, self.in_flight)
        try:
            out = self._next(body)
            await asyncio.sleep(out.get("_delay", self.latency))
        finally:
            self.in_flight -= 1
        return self._response(body, out, cache_hit)


# USD per million tokens: (base input, 5-minute cache write, cache read, output).
# Cache reads cost 0.1x base input, except 0.05x on Opus 5.5 and Sonnet 5.5 (0.025x on Fable 5.1).
PRICES = {
    "claude-sonnet-5-5": (2.00, 2.50, 0.10, 10.00),
    "claude-haiku-5-5": (0.10, 0.125, 0.01, 0.50),    # prompts up to 100k tokens; longer prompts cost more
    "claude-opus-5-5": (4.00, 5.00, 0.20, 20.00),
    "claude-haiku-4-5": (1.00, 1.25, 0.10, 5.00),     # legacy model
}


def cost_usd(model: str, usage) -> float:
    """Dollar cost of one response's usage (unknown models are priced like Sonnet 5.5)."""
    p_in, p_write, p_read, p_out = PRICES.get(model, PRICES["claude-sonnet-5-5"])
    u = usage if isinstance(usage, dict) else usage.model_dump()
    return ((u.get("input_tokens") or 0) * p_in + (u.get("cache_creation_input_tokens") or 0) * p_write
            + (u.get("cache_read_input_tokens") or 0) * p_read + (u.get("output_tokens") or 0) * p_out) / 1e6


print("offline toolkit ready")

## 8.1 Fixed pipelines (prompt chaining)

A fixed pipeline decides every step before the first call. Each call does one focused job, **your code** owns the control flow, and programmatic **gates** (schema checks, business rules, routing branches) sit between steps. Steps that need no model (totals, lookups, formatting) are plain code. [Theory](README.md#81-fixed-pipelines-prompt-chaining)

### A gated invoice chain (offline)

The same code as the README, with two small changes for testability: `step()` takes the client as a parameter (so the fake can be injected) and records a per-step log.

In [ ]:
class Metadata(BaseModel):
    doc_type: Literal["invoice", "receipt", "contract", "other"]
    language: str
    page_count: int | None


class InvoiceData(BaseModel):
    vendor: str | None
    invoice_number: str | None
    line_item_amounts: list[float]
    stated_total: float | None


def step(llm, system: str, user: str, schema: type[BaseModel], log: list | None = None, model: str = MODEL):
    response = llm.messages.parse(
        model=model, max_tokens=1024, system=system,
        messages=[{"role": "user", "content": user}],
        output_format=schema,
    )
    result = parsed_or_raise(response)
    if log is not None:                                 # per-step log = per-step evaluation later
        log.append({"step": schema.__name__, "output": result.model_dump()})
    return result


def invoice_pipeline(document: str, llm=client, log: list | None = None) -> dict:
    meta = step(llm, "Classify the document. Use 'other' if unsure.", document, Metadata, log)
    if meta.doc_type != "invoice":                      # gate 1: code decides the route
        return {"status": "routed", "reason": f"doc_type={meta.doc_type}"}

    data = step(llm, "Extract the invoice fields. Use null when a value is not printed.", document, InvoiceData, log)
    total = round(sum(data.line_item_amounts), 2)       # gate 2: deterministic business rule
    if data.stated_total is not None and abs(total - data.stated_total) > 0.01:
        return {"status": "needs_review", "reason": f"items sum to {total}, stated {data.stated_total}"}

    return {"status": "ok", "data": data.model_dump(), "computed_total": total}   # enrichment is plain code


DOCS = {
    "receipt": "CAFE UNO  receipt #4471\n1 flat white 3.80\nCARD  3.80\nThank you!",
    "invoice_ok": "ACME Supplies Ltd. INVOICE INV-2207\nWidgets x2 ..... 40.00\nShipping ...... 7.50\nTOTAL DUE ...... 47.50",
    "invoice_bad": "Northwind Traders INVOICE NW-118\nConsulting ..... 1200.00\nTravel ......... 310.00\nTOTAL DUE ...... 1600.00",
}
# What a model would return for each document (the fake answers by looking at the request).
SCRIPTED = {
    "receipt": (Metadata(doc_type="receipt", language="en", page_count=1), None),
    "invoice_ok": (Metadata(doc_type="invoice", language="en", page_count=1),
                   InvoiceData(vendor="ACME Supplies Ltd.", invoice_number="INV-2207",
                               line_item_amounts=[40.0, 7.5], stated_total=47.5)),
    "invoice_bad": (Metadata(doc_type="invoice", language="en", page_count=1),
                    InvoiceData(vendor="Northwind Traders", invoice_number="NW-118",
                                line_item_amounts=[1200.0, 310.0], stated_total=1600.0)),
}


def invoice_responder(body: dict) -> dict:
    doc_key = next(k for k, d in DOCS.items() if d in user_text(body))
    meta, data = SCRIPTED[doc_key]
    return reply_json(meta if system_text(body).startswith("Classify") else data)


for key, document in DOCS.items():
    fake, log = FakeClaude(invoice_responder), []
    result = invoice_pipeline(document, llm=fake.client, log=log)
    print(f"{key:12} -> {result['status']:13} calls={len(fake.requests)}  {result.get('reason', '')}")

> **Observe:** the receipt costs **one** call (gate 1 stops the chain), and the mismatched invoice is caught by a line of Python, not by asking a model to "double-check". The gate decides on structured fields (`doc_type`, numbers), never on prose.

### Per-step evaluation (offline)

Because each step is its own call, you can evaluate it alone. Here is a tiny labeled set for the classifier step only. In a real project this is how you find out *which* step is weak before touching the prompt.

In [ ]:
CLASSIFIER_EVAL = [(DOCS["receipt"], "receipt"), (DOCS["invoice_ok"], "invoice"), (DOCS["invoice_bad"], "invoice")]

fake = FakeClaude(invoice_responder)
hits = [step(fake.client, "Classify the document. Use 'other' if unsure.", doc, Metadata).doc_type == label
        for doc, label in CLASSIFIER_EVAL]
print(f"classifier step accuracy: {sum(hits)}/{len(hits)}")

> **Try it:** delete gate 2 (the total check) and rerun the loop. `invoice_bad` now comes out as `"ok"`: a chain without gates passes a bad step straight to the next one. That is the "chains without gates" anti-pattern from the README.

In [ ]:
# (live) the same chain on the real model: expect "needs_review" (1200 + 310 != 1600)
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    log = []
    print(invoice_pipeline(DOCS["invoice_bad"], llm=client, log=log))
    for entry in log:
        print(" ", entry)

## 8.2 Dynamic adaptive decomposition

When the scope is unknown, the subtasks come **from what you discover**. A planner call reads the goal plus compact findings and returns the next subtasks as structured output; your code runs them, records results and asks again. Completion is a **boolean field**, the round cap is a safety net, and a coverage check guards against the silent failure mode: a decomposition that is too narrow. [Theory](README.md#82-dynamic-adaptive-decomposition)

### A toy legacy codebase and an executor (offline)

To make subtasks executable without a worker model, this notebook's `Subtask` adds two fields to the README's version: `action` (one of five tool-like steps) and `argument`. `run_subtask()` executes them against an in-memory repo with real `glob` and `grep` logic. Writing tests is simulated, but it **refuses** to test a module that reads the system clock directly until that module has an injectable clock. That is the discovery the plan must adapt to (the README's `billing` example).

In [ ]:
LEGACY_REPO = {
    "app/billing/invoices.py": "from datetime import datetime\n\ndef late_fee(invoice):\n    days_late = (datetime.now() - invoice.due_date).days\n    return 25.0 if days_late > 30 else 0.0",
    "app/billing/payments.py": "from app.billing.invoices import late_fee\n\ndef amount_due(invoice):\n    return invoice.total + late_fee(invoice)",
    "app/users/accounts.py": "def create_account(email):\n    ...\n\ndef close_account(user_id):\n    ...",
    "app/notifications/reminders.py": "def reminder_text(invoice):\n    ...",
    "app/exports/csv_export.py": "def export_invoices(rows):\n    ...",
    "app/reports/monthly.py": "def monthly_totals(rows):\n    ...",
    "tests/test_smoke.py": "from app.reports.monthly import monthly_totals\n\ndef test_imports(): ...",
}
CHANGES_LAST_90_DAYS = {"app/billing/invoices.py": 38, "app/billing/payments.py": 21, "app/users/accounts.py": 6,
                        "app/notifications/reminders.py": 4, "app/reports/monthly.py": 3, "app/exports/csv_export.py": 2}
CLOCK_READY: set[str] = set()       # modules that already take an injectable clock
CLOCK_CALL = re.compile(r"\b(datetime\.now|datetime\.utcnow|date\.today|time\.time)\(")


class Subtask(BaseModel):
    goal: str                       # self-contained instruction
    rationale: str                  # why this is next: risk, dependency, coverage gap
    priority: Literal["high", "medium", "low"]
    action: Literal["glob", "grep", "read", "inject_clock", "write_tests"]   # added here so code can execute it
    argument: str                   # glob pattern, regex, file path or module directory


class Plan(BaseModel):
    done: bool                      # explicit, structured completion signal
    next_subtasks: list[Subtask]
    open_questions: list[str]


def run_subtask(sub: Subtask) -> str:
    """Execute one subtask against LEGACY_REPO. Returns a short, compact result string."""
    if sub.action == "glob":
        return json.dumps(sorted(p for p in LEGACY_REPO if fnmatch(p, sub.argument)))
    if sub.action == "grep":
        rx = re.compile(sub.argument)
        hits = [f"{p}:{i}: {line.strip()}" for p, src in LEGACY_REPO.items()
                for i, line in enumerate(src.splitlines(), 1) if rx.search(line)]
        return json.dumps(hits[:20])
    if sub.action == "read":
        return LEGACY_REPO.get(sub.argument, f"not found: {sub.argument}")[:400]
    module = sub.argument.rstrip("/")
    if sub.action == "inject_clock":
        CLOCK_READY.add(module)
        return f"{module} now takes a `now` argument (default datetime.now); tests can pass a fixed time"
    # write_tests: blocked if the module reads the system clock directly and has no injectable clock yet
    module_files = [p for p in LEGACY_REPO if p.startswith(module + "/")]
    clock_files = [p for p in module_files if CLOCK_CALL.search(LEGACY_REPO[p])]
    if clock_files and module not in CLOCK_READY:
        return f"BLOCKED: {module} reads the system clock in {clock_files}; add an injectable clock first"
    return f"wrote {3 * len(module_files)} tests for {module} ({len(module_files)} file(s))"


print(run_subtask(Subtask(goal="map", rationale="", priority="high", action="glob", argument="app/**/*.py")))

### The planner loop, driven by a scripted planner (offline)

`scripted_planner` plays the model: it reads the `<findings>` in each request and decides what to do next, the way a real planner would. Watch how the plan changes between rounds.

In [ ]:
PLANNER_SYSTEM = """You plan an investigation step by step.
Given the goal and the findings so far, return the next 1-3 subtasks, highest impact first.
Map structure before acting. When a finding reveals a dependency, schedule the dependency first.
Each subtask uses one action: glob (pattern), grep (regex), read (file path), inject_clock (module directory),
write_tests (module directory such as app/billing).
Set done=true only when the goal is met or no useful subtask remains."""


def adaptive_investigation(goal: str, run_subtask: Callable[[Subtask], str], llm=client,
                           max_rounds: int = 8, verbose: bool = True) -> list[dict]:
    findings: list[dict] = []
    for round_no in range(1, max_rounds + 1):   # safety net, not the primary stop signal
        plan = step(llm, PLANNER_SYSTEM,
                    f"<goal>{goal}</goal>\n<findings>{json.dumps(findings)}</findings>", Plan)
        if verbose:
            print(f"round {round_no}: done={plan.done} next={[f'{s.action}({s.argument})' for s in plan.next_subtasks]}")
        if plan.done or not plan.next_subtasks:
            return findings
        for sub in plan.next_subtasks[:3]:      # cap the fan-out per round
            findings.append({"subtask": sub.goal, "result": run_subtask(sub)})
    findings.append({"status": "budget_exhausted", "note": "report partial results and open questions"})
    return findings


def _sub(goal, action, argument, priority="high", rationale=""):
    return Subtask(goal=goal, rationale=rationale or goal, priority=priority, action=action, argument=argument)


def scripted_planner(body: dict) -> dict:
    """Stands in for the planner model: decides the next subtasks from the findings in the request."""
    findings = json.loads(re.search(r"<findings>(.*)</findings>", user_text(body), re.S).group(1))
    log = " | ".join(f"{f.get('subtask')} => {f.get('result')}" for f in findings)
    nxt: list[Subtask] = []
    if "Map source modules" not in log:
        nxt = [_sub("Map source modules", "glob", "app/**/*.py"), _sub("Map test files", "glob", "tests/*.py")]
    elif "Find which modules tests import" not in log:
        nxt = [_sub("Find which modules tests import", "grep", r"^from app\.")]
    elif "Write tests for app/billing" not in log:
        nxt = [_sub("Write tests for app/billing", "write_tests", "app/billing",
                    rationale="untested, moves money, highest churn (38 + 21 changes in 90 days)")]
    elif "BLOCKED" in log and "Add an injectable clock to app/billing" not in log:
        nxt = [_sub("Add an injectable clock to app/billing", "inject_clock", "app/billing",
                    rationale="late fees depend on the current time (discovered); make time controllable first"),
               _sub("Write tests for app/billing (retry)", "write_tests", "app/billing")]
    elif "Write tests for app/users" not in log:
        nxt = [_sub("Write tests for app/users", "write_tests", "app/users", priority="medium",
                    rationale="untested, next-highest churn")]
    else:
        return reply_json(Plan(done=True, next_subtasks=[], open_questions=[
            "app/notifications and app/exports are still untested (low churn)", "app/reports has only a smoke test"]))
    return reply_json(Plan(done=False, next_subtasks=nxt, open_questions=[]))


CLOCK_READY.clear()
fake = FakeClaude(scripted_planner)
findings = adaptive_investigation("Add comprehensive tests to this legacy codebase", run_subtask, llm=fake.client)
print()
for f in findings:
    print(f"- {f['subtask']}\n    {f['result'][:110]}")

> **Observe:** the "Add an injectable clock to app/billing" subtask did not exist in round 1. It appeared only after `write_tests` reported that billing reads the system clock, and the planner scheduled that dependency **first**. Also notice what each round sends: the goal plus compact findings, not a growing transcript, and the loop ends on `done=True`, not on the word "done" in some text.

### Budget exhaustion is reported, not hidden (offline)

In [ ]:
never_done = FakeClaude(lambda body: reply_json(Plan(done=False, open_questions=[], next_subtasks=[
    _sub("Look around again", "glob", "app/*")])))
out = adaptive_investigation("Find every flaky test", run_subtask, llm=never_done.client, max_rounds=3, verbose=False)
print(f"{len(never_done.requests)} planner calls; last entry: {out[-1]}")

### Guarding against a too-narrow decomposition (offline)

Every worker can succeed while the overall answer still has a hole. A **coverage map** (which subtask covers which aspect of the goal) makes the hole visible before you spend money on the fan-out. Where do the required aspects come from? From your rubric or acceptance criteria, or from a separate critic call that has not seen the planner's reasoning.

In [ ]:
class CoverageItem(BaseModel):    # a list of objects, not dict[str, str]: structured outputs need closed objects
    aspect: str
    subtask_id: str


def coverage_gaps(required_aspects: list[str], coverage: list[CoverageItem]) -> list[str]:
    covered = {c.aspect.strip().lower() for c in coverage}
    return [a for a in required_aspects if a.lower() not in covered]


REQUIRED = ["visual art", "graphic design", "photography", "music", "film and video", "writing and publishing"]
narrow_plan = [CoverageItem(aspect="visual art", subtask_id="t1"), CoverageItem(aspect="graphic design", subtask_id="t2"),
               CoverageItem(aspect="photography", subtask_id="t3")]
gaps = coverage_gaps(REQUIRED, narrow_plan)
print("gaps before fan-out:", gaps)
print("feedback for a re-plan:", f"Your decomposition does not cover: {', '.join(gaps)}. Add subtasks for them.")


# The same check as a critic call: a separate request that sees the goal and the plan, not the planner's reasoning.
class CoverageCritique(BaseModel):
    missing_aspects: list[str]
    overlapping_subtasks: list[str]


critic = FakeClaude([reply_json(CoverageCritique(missing_aspects=["music", "film and video", "writing and publishing"],
                                                 overlapping_subtasks=[]))])
critique = step(critic.client, "List major sub-domains of the goal that the plan does not cover, and overlapping subtasks.",
                f"<goal>Impact of AI on creative industries</goal>\n<plan>{[c.model_dump() for c in narrow_plan]}</plan>",
                CoverageCritique)
print("critic says missing:", critique.missing_aspects)

> **Try it:** change `scripted_planner` so that it never schedules the clock (skip that branch). The loop still terminates, but the billing tests stay `BLOCKED` in the findings. A good final report must list that as an open item rather than claim full coverage.

In [ ]:
# (live) the real model plans the same investigation; run_subtask executes its subtasks offline.
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    CLOCK_READY.clear()
    findings = adaptive_investigation("Add comprehensive tests to this legacy codebase. Prioritize by risk "
                                      "(see change counts) and coverage gaps.\nChange counts (90 days): "
                                      + json.dumps(CHANGES_LAST_90_DAYS), run_subtask, llm=client, max_rounds=6)
    for f in findings:
        print("-", f.get("subtask", f.get("status")), "=>", str(f.get("result", f.get("note")))[:100])

## 8.3 Multi-pass code review

One call over a large PR dilutes attention: uneven depth, contradictory findings, missed obvious bugs. The fix is one **local pass per file** with a shared rubric and schema, then one **integration pass** that receives only interfaces, data flow and local findings and reports only cross-file issues. Merge and dedupe in code. [Theory](README.md#83-multi-pass-code-review)

### A seeded 8-file PR (offline)

Eight small files with **planted** bugs. `SEEDED_BUGS` is the answer key: four local bugs, three cross-file bugs, and one repeated pattern (float money rounding in `pricing.py` and `discounts.py`) that a consistent reviewer flags in both files or in neither.

In [ ]:
SEEDED_PR = {
    "inventory/models.py": """from dataclasses import dataclass

@dataclass
class Item:
    sku: str
    price: float
    available: int

_ITEMS: dict[str, Item] = {}

def get_item(sku: str) -> Item | None:
    return _ITEMS.get(sku)          # None for unknown SKUs
""",
    "inventory/stock.py": """from inventory.models import get_item

def reserve(sku: str, qty: int) -> dict:
    item = get_item(sku)
    if item.available < qty:
        raise ValueError("insufficient stock")
    item.available -= qty
    return {"sku": sku, "reserved": qty}
""",
    "inventory/pricing.py": """def unit_total(item, qty: int) -> float:
    price = item.price * qty
    return round(price * 100) / 100
""",
    "inventory/discounts.py": """def apply_discount(total: float, pct: int) -> float:
    total = total * (1 - pct / 100)
    return round(total * 100) / 100
""",
    "inventory/errors.py": """class OutOfStock(Exception):
    pass
""",
    "api/routes.py": """from inventory.stock import reserve

def post_reservation(payload: dict) -> dict:
    result = reserve(payload["qty"], payload["sku"])
    return {"ok": True, "reservation_id": result["id"]}
""",
    "orders/service.py": """from inventory.stock import reserve
from inventory.errors import OutOfStock

def place_order(db, sku: str, qty: int) -> dict:
    try:
        reserve(sku, qty)
    except OutOfStock:
        return {"status": "backordered"}
    db.execute(f"INSERT INTO orders (sku, qty) VALUES ('{sku}', {qty})")
    return {"status": "placed"}
""",
    "reports/daily.py": """def average_order_value(orders: list[dict]) -> float:
    return sum(o["total"] for o in orders) / len(orders)
""",
}

SEEDED_BUGS = [  # id, file, kind, keywords that identify a matching finding
    ("L1", "inventory/stock.py", "local", ["none"]),
    ("L2", "inventory/discounts.py", "local", ["pct", "percent", "negative"]),
    ("L3", "orders/service.py", "local", ["sql", "inject"]),
    ("L4", "reports/daily.py", "local", ["empty", "zero"]),
    ("X1", "api/routes.py", "cross", ["order", "swap", "argument"]),
    ("X2", "api/routes.py", "cross", ["'id'", '"id"', "reservation_id", "no id", "keyerror"]),
    ("X3", "orders/service.py", "cross", ["outofstock", "valueerror"]),
]
REPEATED_PATTERN = (["inventory/pricing.py", "inventory/discounts.py"], ["float", "round", "decimal"])
print({p: len(s.splitlines()) for p, s in SEEDED_PR.items()})

In [ ]:
class Finding(BaseModel):
    file: str
    line: int | None
    severity: Literal["critical", "high", "medium", "low"]
    issue: str
    suggested_fix: str
    evidence: str                   # the exact code the finding is about


class FileReview(BaseModel):
    findings: list[Finding]
    exports: list[str]              # public symbols with signatures; feeds the integration pass
    imports: list[str]


class CrossFileReview(BaseModel):
    findings: list[Finding]         # cross-file issues only (also used as the single-pass schema)


LOCAL_RULES = """Review ONE file. Report only issues visible inside this file.
Use the severity rubric exactly. Quote the code each finding is about in `evidence`.
List the file's public symbols with signatures in `exports` and its imports in `imports`."""

INTEGRATION_RULES = """You receive each file's public interface and its local findings.
Report ONLY issues that cross file boundaries: contracts, types, data flow, error propagation,
circular dependencies. Do not repeat local findings."""

SINGLE_PASS_RULES = "Review this pull request. Report every issue you find, with evidence."


def integration_summary(local: dict[str, FileReview]) -> str:
    return "\n".join(
        f'<file path="{p}">\n<exports>{r.exports}</exports>\n<imports>{r.imports}</imports>\n'
        f"<local_findings>{[f.issue for f in r.findings]}</local_findings>\n</file>"
        for p, r in local.items()
    )


async def review_file(llm, path: str, source: str, sem: asyncio.Semaphore) -> tuple[str, FileReview]:
    async with sem:
        response = await llm.messages.parse(
            model=MODEL, max_tokens=2048, system=LOCAL_RULES,
            messages=[{"role": "user", "content": f'<file path="{path}">\n{source}\n</file>'}],
            output_format=FileReview,
        )
    return path, parsed_or_raise(response)


async def multi_pass_review(files: dict[str, str], llm=aclient, concurrency: int = 5) -> dict:
    sem = asyncio.Semaphore(concurrency)
    local = dict(await asyncio.gather(*(review_file(llm, p, s, sem) for p, s in files.items())))
    integration = parsed_or_raise(await llm.messages.parse(
        model=MODEL, max_tokens=2048, system=INTEGRATION_RULES,
        messages=[{"role": "user", "content": integration_summary(local)}], output_format=CrossFileReview,
    ))
    return {"local": local, "cross_file": integration.findings}


async def single_pass_review(files: dict[str, str], llm=aclient) -> list[Finding]:
    pr = "\n".join(f'<file path="{p}">\n{s}\n</file>' for p, s in files.items())
    response = await llm.messages.parse(model=MODEL, max_tokens=4096, system=SINGLE_PASS_RULES,
                                        messages=[{"role": "user", "content": pr}], output_format=CrossFileReview)
    return parsed_or_raise(response).findings


def dedupe(findings: list[Finding]) -> list[Finding]:
    seen, out = set(), []
    for f in findings:
        key = (f.file, f.line, re.sub(r"\W+", " ", f.issue.lower()).strip()[:60])
        if key not in seen:
            seen.add(key)
            out.append(f)
    return out


def review_metrics(findings: list[Finding]) -> dict:
    """Recall against SEEDED_BUGS, findings per file, and the repeated-pattern consistency check."""
    def text(f):
        return f"{f.issue} {f.evidence} {f.suggested_fix}".lower()
    found = {bug_id for bug_id, path, _, kws in SEEDED_BUGS
             if any(f.file == path and any(k in text(f) for k in kws) for f in findings)}
    files, kws = REPEATED_PATTERN
    flagged = [p for p in files if any(f.file == p and any(k in text(f) for k in kws) for f in findings)]
    return {
        "local_recall": f"{sum(b[0] in found for b in SEEDED_BUGS if b[2] == 'local')}/4",
        "cross_recall": f"{sum(b[0] in found for b in SEEDED_BUGS if b[2] == 'cross')}/3",
        "missed": sorted({b[0] for b in SEEDED_BUGS} - found),
        "per_file": dict(Counter(f.file for f in findings)),
        "pattern_consistent": len(flagged) in (0, len(files)),
    }


print("review functions ready")

### Single pass vs multi-pass on the fake (offline)

The single-pass reply below is **scripted to show the symptoms** the README describes (deep comments on the first files, nothing on the last ones, the money-rounding pattern flagged in one file and not the other). The per-file replies are what a focused pass with a shared rubric tends to return. The point of this cell is the **measurement machinery**; the live cell after it measures real behavior.

In [ ]:
def F(file, line, sev, issue, evidence, fix="see issue"):
    return Finding(file=file, line=line, severity=sev, issue=issue, evidence=evidence, suggested_fix=fix)


SINGLE_PASS_SCRIPT = CrossFileReview(findings=[
    F("inventory/models.py", 11, "low", "get_item returns None for unknown SKUs; document it", "return _ITEMS.get(sku)"),
    F("inventory/stock.py", 5, "high", "item may be None when the SKU is unknown (AttributeError)", "if item.available < qty:"),
    F("inventory/pricing.py", 3, "medium", "float money with round(); use Decimal or integer cents", "round(price * 100) / 100"),
    F("api/routes.py", 4, "high", "reserve() is called with arguments in the wrong order (qty, sku)", 'reserve(payload["qty"], payload["sku"])'),
])

LOCAL_SCRIPT = {
    "inventory/models.py": [],
    "inventory/stock.py": [F("inventory/stock.py", 5, "high", "item is None for unknown SKUs (AttributeError)", "if item.available < qty:")],
    "inventory/pricing.py": [F("inventory/pricing.py", 3, "medium", "float money rounding; use Decimal or cents", "round(price * 100) / 100")],
    "inventory/discounts.py": [F("inventory/discounts.py", 3, "medium", "float money rounding; use Decimal or cents", "round(total * 100) / 100"),
                               F("inventory/discounts.py", 2, "high", "pct is not validated; pct > 100 gives a negative total", "total * (1 - pct / 100)")],
    "inventory/errors.py": [],
    "api/routes.py": [],
    "orders/service.py": [F("orders/service.py", 9, "critical", "SQL injection: f-string query with user input", "db.execute(f\"INSERT ...")],
    "reports/daily.py": [F("reports/daily.py", 2, "high", "ZeroDivisionError on an empty list of orders", "/ len(orders)")],
}
EXPORTS = {"inventory/stock.py": ["reserve(sku: str, qty: int) -> dict  # keys: sku, reserved; raises ValueError"],
           "inventory/errors.py": ["class OutOfStock(Exception)"],
           "api/routes.py": ["post_reservation(payload: dict) -> dict"],
           "orders/service.py": ["place_order(db, sku: str, qty: int) -> dict"]}
CROSS_SCRIPT = CrossFileReview(findings=[
    F("api/routes.py", 4, "high", "reserve(sku, qty) is called as reserve(qty, sku): arguments swapped", 'reserve(payload["qty"], payload["sku"])'),
    F("api/routes.py", 5, "high", "reserve() returns no 'id' key, so result['id'] raises KeyError", 'result["id"]'),
    F("orders/service.py", 7, "high", "reserve raises ValueError, but place_order catches OutOfStock, so backorders never happen", "except OutOfStock:"),
])


def review_responder(body: dict) -> dict:
    sys_prompt = system_text(body)
    if sys_prompt == SINGLE_PASS_RULES:
        return reply_json(SINGLE_PASS_SCRIPT)
    if sys_prompt == INTEGRATION_RULES:
        return reply_json(CROSS_SCRIPT)
    path = re.search(r'<file path="([^"]+)">', user_text(body)).group(1)
    return reply_json(FileReview(findings=LOCAL_SCRIPT[path], exports=EXPORTS.get(path, []), imports=[]))


fake = FakeClaude(review_responder, latency=0.02)
single = await single_pass_review(SEEDED_PR, llm=fake.aclient)
multi = await multi_pass_review(SEEDED_PR, llm=fake.aclient)
merged = dedupe([f for r in multi["local"].values() for f in r.findings] + multi["cross_file"])

print("single pass :", review_metrics(single))
print("multi-pass  :", review_metrics(merged))
print(f"\ncalls: {len(fake.requests)} (1 single-pass + {len(SEEDED_PR)} local + 1 integration)")
source_chars = sum(len(s) for s in SEEDED_PR.values())
summary_chars = len(user_text(fake.requests[-1]))
print(f"integration input: {summary_chars} chars of interfaces + findings vs {source_chars} chars of source")

> **Observe:** three things the metrics make visible. `per_file` shows the uneven depth of the single pass (nothing on the last files). `pattern_consistent=False` is the "flagged here, approved there" contradiction, which the shared rubric removes. And the integration pass receives interfaces and findings, not the 8 files again. Here the files are only a few lines each, so the summary is about as long as the source; its size grows with the number of **interfaces and findings**, not with lines of code, so on a real 14-file PR with hundreds of lines per file it is a small fraction of the source.

> **Try it:** add a voting baseline. Script three single-pass replies that each catch a *different* subset of the bugs, keep only issues found in at least two of them, and run `review_metrics` on the result. Inconsistently detected real bugs get voted out, which is why the README rejects majority voting as the fix for attention dilution.

In [ ]:
# (live) measure it for real: one single pass vs 8 local passes + 1 integration pass (~10 calls)
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    live_single = await single_pass_review(SEEDED_PR)
    live_multi = await multi_pass_review(SEEDED_PR)
    live_merged = dedupe([f for r in live_multi["local"].values() for f in r.findings] + live_multi["cross_file"])
    print("single pass :", review_metrics(live_single))
    print("multi-pass  :", review_metrics(live_merged))
    for f in live_multi["cross_file"]:
        print(f"  cross-file [{f.severity}] {f.file}:{f.line} {f.issue}")

## 8.4 Independent review instances: why self-review is weak

A model reviewing its own work **in the same conversation** still carries the reasoning that produced it, so it tends to confirm it. The fix is a **second, independent instance**: a fresh request with the artifact, the spec and the criteria, and nothing of the generator's conversation. With a stateless API, independence is simply what you put in `messages`. [Theory](README.md#84-independent-review-instances-why-self-review-is-weak)

### A generator that talked itself into a bug (offline)

The spec says the last chunk may be shorter. The generated code silently drops it, and the generator's own reasoning says it checked exactly that case. First, prove the bug is real by running the code.

In [ ]:
SPEC = ("Write chunk(items, size) that splits a list into consecutive chunks of length `size`. "
        "The last chunk may be shorter. Raise ValueError if size < 1.")
GENERATED_DIFF = """def chunk(items, size):
    if size < 1:
        raise ValueError("size must be >= 1")
    return [items[i:i + size] for i in range(0, len(items) - size + 1, size)]
"""
AUTHOR_RATIONALE = ("I considered the case where len(items) is not a multiple of size; "
                    "the range bound handles it, so the approach is fine.")
GENERATOR_MESSAGES = [
    {"role": "user", "content": SPEC},
    {"role": "assistant", "content": f"{AUTHOR_RATIONALE}\n\n```python\n{GENERATED_DIFF}```"},
]
CRITERIA = "Correctness against the spec, including edge cases (empty list, partial last chunk, size larger than the list)."

namespace: dict = {}
exec(GENERATED_DIFF, namespace)
print("chunk([1, 2, 3, 4, 5], 2) =", namespace["chunk"]([1, 2, 3, 4, 5], 2), " <- the 5 is gone")

In [ ]:
class Review(BaseModel):
    approved: bool
    findings: list[Finding]


REVIEWER_SYSTEM = """You are a code reviewer who did not write this change.
Check it against the spec and the criteria. Assume nothing is correct until you have checked it.
Report only issues you can point to in the diff."""


def self_review_request(generator_messages: list[dict]) -> dict:
    """ANTI-PATTERN: the reviewer inherits the author's reasoning."""
    return dict(model=MODEL, max_tokens=2048, output_format=Review,
                messages=generator_messages + [{"role": "user", "content": "Now critically review your code."}])


def independent_review_request(spec: str, diff: str, criteria: str, model: str = MODEL) -> dict:
    """A brand-new request: no generator messages, no thinking blocks, no author rationale."""
    return dict(model=model, max_tokens=2048, system=REVIEWER_SYSTEM, output_format=Review,
                messages=[{"role": "user", "content":
                           f"<spec>\n{spec}\n</spec>\n<criteria>\n{criteria}\n</criteria>\n<diff>\n{diff}\n</diff>"}])


def leaked_context(request: dict, generator_messages: list[dict]) -> list[str]:
    """Sentences of the generator's own (assistant) messages that reach the reviewer."""
    sent = json.dumps(request["messages"])
    author_text = " ".join(m["content"] if isinstance(m["content"], str) else json.dumps(m["content"])
                           for m in generator_messages if m["role"] == "assistant")
    sentences = [s.strip() for s in re.split(r"(?<=[.;])\s+", author_text.split("```")[0]) if len(s.strip()) > 20]
    return [s for s in sentences if json.dumps(s)[1:-1] in sent]


for name, req in [("self-review", self_review_request(GENERATOR_MESSAGES)),
                  ("independent", independent_review_request(SPEC, GENERATED_DIFF, CRITERIA))]:
    roles = [m["role"] for m in req["messages"]]
    print(f"{name:12} roles={roles}\n{'':12} leaked author reasoning: {leaked_context(req, GENERATOR_MESSAGES)}")

### What the two reviewers say (offline, illustrative)

The replies below are scripted to illustrate the typical outcome; the live cell checks it for real. What matters offline is the wiring: the same `messages.parse` call, with very different inputs.

In [ ]:
def review_script(body: dict) -> dict:
    if body.get("system") == REVIEWER_SYSTEM:
        return reply_json(Review(approved=False, findings=[Finding(
            file="chunk.py", line=4, severity="high", evidence="range(0, len(items) - size + 1, size)",
            issue="The partial last chunk is dropped: chunk([1,2,3,4,5], 2) returns [[1,2],[3,4]], violating the spec.",
            suggested_fix="Use range(0, len(items), size).")]))
    return reply_json(Review(approved=True, findings=[]))    # the author agrees with itself


fake = FakeClaude(review_script)
for name, req in [("self-review", self_review_request(GENERATOR_MESSAGES)),
                  ("independent", independent_review_request(SPEC, GENERATED_DIFF, CRITERIA))]:
    review = parsed_or_raise(fake.client.messages.parse(**req))
    print(f"{name:12} approved={review.approved} findings={[f.issue[:70] for f in review.findings]}")

> **Observe:** the only difference between the two calls is the **input**. Same model, same schema, same SDK method. Independence is an architecture decision, not a prompt trick. Optional strengtheners: a fault-finding system prompt (done here), a different model for the reviewer (`independent_review_request(..., model=...)`), and a rule that every finding cites evidence from the diff.

> **Try it:** add `"Think carefully and be very critical."` to the self-review message and rerun the live cell a few times. More effort inside the same context is not the same as a fresh pair of eyes.

In [ ]:
# (live) both reviewers on the real model
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    for name, req in [("self-review", self_review_request(GENERATOR_MESSAGES)),
                      ("independent", independent_review_request(SPEC, GENERATED_DIFF, CRITERIA))]:
        review = parsed_or_raise(client.messages.parse(**req))
        print(f"{name:12} approved={review.approved}")
        for f in review.findings:
            print(f"{'':12} [{f.severity}] {f.issue[:110]}")

## 8.5 Verification passes and confidence-based routing

Review passes produce **candidates**. A verification pass (an independent instance) re-checks each one and returns a verdict, evidence and a **confidence level per finding**. That field drives routing, but only after you **calibrate** it: measure precision per confidence bucket on labeled findings and auto-post only the buckets that meet your target. Keep sampling to catch drift. [Theory](README.md#85-verification-passes-and-confidence-based-routing)

In [ ]:
class Verdict(BaseModel):
    finding_id: str
    verdict: Literal["confirmed", "plausible", "refuted"]
    evidence: str                   # file:line and the code that proves or disproves it
    confidence: Literal["high", "medium", "low"]


VERIFIER_SYSTEM = """You verify ONE candidate code-review finding against the source.
Decide confirmed / plausible / refuted, quote the code that proves it, and rate your confidence.
Refute findings that the code does not support."""


async def verify(llm, finding_id: str, finding: Finding, source: str, sem: asyncio.Semaphore) -> Verdict:
    async with sem:
        response = await llm.messages.parse(
            model=MODEL, max_tokens=1024, system=VERIFIER_SYSTEM, output_format=Verdict,
            messages=[{"role": "user", "content": f"<finding id=\"{finding_id}\">{finding.model_dump_json()}</finding>\n"
                                                  f"<source path=\"{finding.file}\">\n{source}\n</source>"}],
        )
    return parsed_or_raise(response)


def route(v: Verdict, auto_post: set[str]) -> str:
    """auto_post comes from calibration on labeled data, e.g. {"high"} once high reaches the precision target."""
    if v.verdict == "refuted":
        return "drop_and_log"
    if v.verdict == "confirmed" and v.confidence in auto_post:
        return "post_inline"
    return "human_triage"


def precision_by_bucket(labeled: list[tuple[str, bool]]) -> dict[str, tuple[float, int]]:
    """labeled = [(confidence, was_real_issue), ...] -> {bucket: (precision, n)}"""
    buckets: dict[str, list[bool]] = {}
    for confidence, real in labeled:
        buckets.setdefault(confidence, []).append(real)
    return {c: (round(sum(v) / len(v), 3), len(v)) for c, v in buckets.items()}


def calibrate(labeled: list[tuple[str, bool]], target: float = 0.95, min_n: int = 30) -> set[str]:
    """Auto-post only buckets that meet the precision target on enough labeled samples."""
    return {c for c, (p, n) in precision_by_bucket(labeled).items() if p >= target and n >= min_n}


print("verification functions ready")

### Calibrate on a labeled history, then route new candidates (offline)

`LABELED` stands for last month's verified findings that engineers marked as real or not real. It is synthetic here but has the shape you would export from your review tool.

In [ ]:
LABELED = ([("high", True)] * 58 + [("high", False)] * 2
           + [("medium", True)] * 36 + [("medium", False)] * 14
           + [("low", True)] * 9 + [("low", False)] * 21)
print("precision by bucket:", precision_by_bucket(LABELED))
AUTO_POST = calibrate(LABELED, target=0.95)
print("auto-post buckets:", AUTO_POST)

# New candidates: the offline multi-pass findings from 8.3, verified in parallel by the (fake) verifier.
offline_findings = dedupe([f for fs in LOCAL_SCRIPT.values() for f in fs] + CROSS_SCRIPT.findings)
candidates = {f"c{i}": f for i, f in enumerate(offline_findings[:6], 1)}
candidates["c7"] = Finding(file="reports/daily.py", line=2, severity="low", issue="sum() is slow for large lists",
                           evidence="sum(o['total'] for o in orders)", suggested_fix="use numpy")
SCRIPTED_VERDICTS = {"c1": ("confirmed", "high"), "c2": ("confirmed", "medium"), "c3": ("confirmed", "high"),
                     "c4": ("plausible", "medium"), "c5": ("confirmed", "high"), "c6": ("confirmed", "low"),
                     "c7": ("refuted", "high")}


def verifier_script(body: dict) -> dict:
    fid = re.search(r'<finding id="(\w+)">', user_text(body)).group(1)
    verdict, conf = SCRIPTED_VERDICTS[fid]
    return reply_json(Verdict(finding_id=fid, verdict=verdict, confidence=conf, evidence=f"{candidates[fid].file}:{candidates[fid].line}"))


fake = FakeClaude(verifier_script, latency=0.02)
sem = asyncio.Semaphore(4)
verdicts = await asyncio.gather(*(verify(fake.aclient, fid, f, SEEDED_PR.get(f.file, ""), sem) for fid, f in candidates.items()))
for v in verdicts:
    print(f"{v.finding_id}: {v.verdict:9} {v.confidence:6} -> {route(v, AUTO_POST)}")
print(Counter(route(v, AUTO_POST) for v in verdicts))

### Why "drop everything below a threshold" is the wrong gate (offline)

Same labeled history, two policies. The uncalibrated policy silently drops the medium and low buckets. Count how many **real** issues that throws away compared with a triage queue that keeps them visible to a human.

In [ ]:
real_dropped = sum(real for conf, real in LABELED if conf != "high")
print(f"'keep only high' silently drops {real_dropped} real issues out of {sum(r for _, r in LABELED)}")
print("calibrated routing keeps them in the triage queue: same noise reduction on the PR, no hidden bugs")


def audit_sample(posted_ids: list[str], every: int = 10) -> list[str]:
    """Deterministic 1-in-N spot check of auto-posted findings, to catch calibration drift."""
    return [fid for i, fid in enumerate(sorted(posted_ids)) if i % every == 0]


posted = [v.finding_id for v in verdicts if route(v, AUTO_POST) == "post_inline"]
print("spot-check these auto-posted findings this week:", audit_sample(posted, every=2))

> **Observe:** confidence is only an **input** to routing, and the threshold is a measured fact about your data, not a guess. The README's exam trap draws exactly this line: uncalibrated self-rated confidence as the only gate is wrong; per-finding confidence calibrated on labeled data and routed to human attention is right.

> **Try it:** change the medium bucket to 48 real / 2 not real (96%, n = 50) and rerun `calibrate`: medium now qualifies. Then cut the medium sample to 20 items (for example 19 real / 1 not real) and see `min_n` keep it out: a precise-looking bucket with too few samples is not evidence.

In [ ]:
# (live) verify three candidates from the seeded PR with the real model
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    sem = asyncio.Semaphore(3)
    real_candidates = {"r1": F("inventory/stock.py", 5, "high", "item may be None for unknown SKUs", "if item.available < qty:"),
                       "r2": F("reports/daily.py", 2, "high", "ZeroDivisionError when orders is empty", "/ len(orders)"),
                       "r3": F("inventory/errors.py", 1, "medium", "OutOfStock should inherit from ValueError", "class OutOfStock(Exception)")}
    for v in await asyncio.gather(*(verify(aclient, k, f, SEEDED_PR[f.file], sem) for k, f in real_candidates.items())):
        print(f"{v.finding_id}: {v.verdict:9} {v.confidence:6} -> {route(v, AUTO_POST):13} | {v.evidence[:80]}")

## 8.6 The workflow pattern catalog

Workflows run LLM calls through **predefined code paths**; agents direct their own process. The catalog: prompt chaining (8.1), **routing**, **parallelization** (sectioning and voting), orchestrator-workers (8.8), **evaluator-optimizer**, and the agent loop. Each is built from one augmented LLM call. This section builds the ones not covered elsewhere. [Theory](README.md#86-the-workflow-pattern-catalog)

### Routing with a cheap classifier and model tiering (offline)

A fast model classifies into **structured fields**, your code dispatches. Always keep an `other` route.

In [ ]:
class Route(BaseModel):
    category: Literal["billing", "technical", "refund", "other"]
    complexity: Literal["simple", "complex"]


ROUTER_SYSTEM = "Classify the support request. Use 'other' when no category fits."


def route_request(query: str, llm=client) -> tuple[str, str]:
    choice = parsed_or_raise(llm.messages.parse(
        model=FAST_MODEL, max_tokens=512, system=ROUTER_SYSTEM,
        messages=[{"role": "user", "content": query}], output_format=Route,
    ))
    model = FAST_MODEL if choice.complexity == "simple" else MODEL    # tiering: cost vs capability
    return choice.category, model


DISPATCH = {   # each route has its own prompt, tools or downstream process
    "billing": "billing flow (invoice tools)",
    "technical": "technical flow (diagnostics tools)",
    "refund": "refund flow (refund tool + approval rule in code)",
    "other": "human queue",
}
QUERIES = {
    "Why was I charged twice this month?": Route(category="billing", complexity="simple"),
    "The SDK times out only when I stream responses over a corporate proxy.": Route(category="technical", complexity="complex"),
    "The kettle arrived broken, I want my money back.": Route(category="refund", complexity="simple"),
    "Do you sponsor student hackathons?": Route(category="other", complexity="simple"),
}
fake = FakeClaude(lambda body: reply_json(QUERIES[user_text(body)]))
for q in QUERIES:
    category, model = route_request(q, llm=fake.client)
    print(f"{q[:55]:57} -> {category:9} {model:18} -> {DISPATCH[category]}")
print("models requested on the wire:", {r["model"] for r in fake.requests})

In [ ]:
# (live) the router on Claude Haiku 5.5: four tiny calls
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    for q in QUERIES:
        print(route_request(q), "<-", q[:60])

### Parallelization, sectioning: answer and screen at the same time (offline)

Two independent calls run concurrently: one answers, one screens the request for a policy problem. Latency is the slower of the two, not the sum.

In [ ]:
class Screen(BaseModel):
    flagged: bool
    reason: str


SCREEN_SYSTEM = "Decide if the request asks for something outside a retail support policy. Explain briefly."


async def answer_and_screen(query: str, llm=aclient) -> dict:
    answer_call = llm.messages.create(model=MODEL, max_tokens=512, messages=[{"role": "user", "content": query}])
    screen_call = llm.messages.parse(model=FAST_MODEL, max_tokens=256, system=SCREEN_SYSTEM, output_format=Screen,
                                     messages=[{"role": "user", "content": query}])
    answer, screen = await asyncio.gather(answer_call, screen_call)
    screen = parsed_or_raise(screen)
    return {"answer": None if screen.flagged else text_of(answer), "screen": screen.model_dump()}


def sectioning_script(body: dict) -> dict:
    if body.get("system") == SCREEN_SYSTEM:
        return reply_json(Screen(flagged=False, reason="ordinary order question")) | {"_delay": 0.3}
    return reply("Your order ships within 2 business days.") | {"_delay": 0.3}


fake = FakeClaude(sectioning_script)
t0 = time.perf_counter()
out = await answer_and_screen("When will my order ship?", llm=fake.aclient)
print(out, f"\nelapsed {time.perf_counter() - t0:.2f}s for two 0.3s calls; peak concurrency {fake.max_in_flight}")

### Parallelization, voting: the threshold picks the trade-off (offline)

Three differently prompted reviewers flagged (1) or passed (0) ten code changes. Ground truth says which changes really contain an injection bug. Compare the vote rules.

In [ ]:
TRUTH = [1, 1, 1, 1, 0, 0, 0, 0, 0, 1]          # change i really has a vulnerability?
VOTES = [                                         # reviewer x change
    [1, 1, 0, 1, 0, 1, 0, 0, 0, 0],
    [1, 0, 1, 1, 0, 0, 0, 1, 0, 1],
    [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
]


def vote(rule: str) -> list[int]:
    need = {"any": 1, "majority": 2, "unanimous": 3}[rule]
    return [int(sum(col) >= need) for col in zip(*VOTES)]


def precision_recall(pred: list[int]) -> tuple[float, float]:
    tp = sum(p and t for p, t in zip(pred, TRUTH))
    return (tp / max(sum(pred), 1), tp / sum(TRUTH))


for rule in ("any", "majority", "unanimous"):
    p, r = precision_recall(vote(rule))
    print(f"{rule:9} precision={p:.2f} recall={r:.2f} flagged={sum(vote(rule))}")

> **Observe:** `any` catches every real vulnerability at the cost of false alarms (add a verification pass after it); `majority` and `unanimous` trade recall for precision. For security screening, missing a bug is the expensive error, so union voting plus verification fits. For the 14-file review problem, voting hides the real cause (attention dilution), which is why the README rejects it there.

### Evaluator-optimizer with a round cap (offline)

One call drafts, a **separate** call evaluates against criteria, and the loop revises until it passes or hits the cap. Non-convergence is reported, not hidden.

In [ ]:
class Evaluation(BaseModel):
    passed: bool
    issues: list[str]                # specific, actionable problems against the criteria


EVAL_SYSTEM = "Evaluate the draft strictly against the criteria. List concrete issues."


def evaluator_optimizer(task: str, criteria: str, llm=client, max_rounds: int = 3) -> dict:
    draft = text_of(llm.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": task}]))
    for round_no in range(1, max_rounds + 1):
        ev = step(llm, EVAL_SYSTEM, f"<task>{task}</task>\n<criteria>{criteria}</criteria>\n<draft>{draft}</draft>", Evaluation)
        if ev.passed:
            return {"draft": draft, "rounds": round_no, "converged": True}
        draft = text_of(llm.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content":
            f"<task>{task}</task>\n<draft>{draft}</draft>\n<issues>{ev.issues}</issues>\nRevise to fix every issue."}]))
    return {"draft": draft, "rounds": max_rounds, "converged": False}   # surface non-convergence, don't hide it


TASK = "Write a one-sentence product tagline for a reusable steel water bottle."
CRITERIA_TAGLINE = "At most 10 words. Mentions durability. No exclamation marks."
converging = FakeClaude([
    reply("Stay hydrated in style with our amazing bottle!"),
    reply_json(Evaluation(passed=False, issues=["does not mention durability", "uses an exclamation mark"])),
    reply("Built to last, refill after refill."),
    reply_json(Evaluation(passed=True, issues=[])),
])
print(evaluator_optimizer(TASK, CRITERIA_TAGLINE, llm=converging.client), "| calls:", len(converging.requests))

stubborn = FakeClaude([reply("Best bottle ever!")] + [reply_json(Evaluation(passed=False, issues=["exclamation mark"])),
                                                     reply("Best bottle ever!!")] * 3)
print(evaluator_optimizer(TASK, CRITERIA_TAGLINE, llm=stubborn.client, max_rounds=3), "| calls:", len(stubborn.requests))

> **Try it:** the evaluator here only sees task, criteria and draft. Change the evaluator call to include the generator's revision request too, and think about why that weakens it (the same reason as 8.4).

In [ ]:
# (live) evaluator-optimizer on the real model (2-6 small calls)
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    print(evaluator_optimizer(TASK, CRITERIA_TAGLINE))

## 8.7 Workflow or agent: choosing the level of autonomy

Pick the simplest structure that works: augmented call, then workflow, then agent, then multi-agent, adding autonomy only when evals show it helps. Multi-agent buys context isolation, parallelism, specialization and independence, but Anthropic's research system measured about **4×** the tokens of chat for an agent and **15×** for multi-agent. Delegate with complete briefs: objective, output format, tools and sources, boundaries. [Theory](README.md#87-workflow-or-agent-choosing-the-level-of-autonomy)

In [ ]:
def recommend_structure(p: dict) -> str:
    """The README's decision flowchart as code. p holds booleans describing the task."""
    if p.get("one_call_enough"):
        return "augmented LLM call"
    if p.get("distinct_categories"):
        return "routing (then the right structure per route)"
    if p.get("steps_known"):
        return "prompt chain with gates" if p.get("steps_depend") else "parallelization: sectioning"
    if p.get("long_exploratory"):
        return "multi-agent (coordinator + subagents)" if p.get("breadth_first_independent") else "agent loop"
    return "orchestrator-workers"


SCENARIOS = {
    "Summarize one support ticket into 3 fields": {"one_call_enough": True},
    "'Autonomous multi-agent system' to sort emails into 4 queues": {"distinct_categories": True},
    "Same 5 clause checks on every contract, each feeding the report": {"steps_known": True, "steps_depend": True},
    "Same rubric on each of 12 independent files": {"steps_known": True, "steps_depend": False},
    "Research report whose sub-questions depend on the request": {},
    "Debug a flaky incident in an unfamiliar system": {"long_exploratory": True},
    "Survey 40 independent sources for a market scan": {"long_exploratory": True, "breadth_first_independent": True},
}
for name, profile in SCENARIOS.items():
    print(f"{name:66} -> {recommend_structure(profile)}")

### What autonomy costs (offline)

A back-of-the-envelope model using the multipliers from Anthropic's research-system write-up. They are averages from one system; measure your own, but the order of magnitude is what decides architecture reviews.

In [ ]:
BASE_IN, BASE_OUT = 6_000, 800             # tokens for one well-prompted chat-style call
MULTIPLIER = {"augmented call": 1, "agent (~4x)": 4, "multi-agent (~15x)": 15}
TASKS_PER_MONTH = 20_000
for name, m in MULTIPLIER.items():
    per_task = cost_usd(MODEL, {"input_tokens": BASE_IN * m, "output_tokens": BASE_OUT * m})
    print(f"{name:20} ${per_task:7.3f} per task  ${per_task * TASKS_PER_MONTH:10,.0f} per month")

### Complete briefs vs one-line delegation (offline)

In [ ]:
BRIEF_FIELDS = ("objective", "output_format", "tools_and_sources", "boundaries")


def brief_problems(brief: dict) -> list[str]:
    problems = [f"missing {k}" for k in BRIEF_FIELDS if not str(brief.get(k, "")).strip()]
    if len(str(brief.get("objective", "")).split()) < 8:
        problems.append("objective is too vague to act on without guessing")
    return problems


vague = {"objective": "Research chip shortages"}
complete = {
    "objective": "Find how the 2021-2023 automotive chip shortage changed carmakers' supplier contracts",
    "output_format": "5-8 bullet findings, each with source URL and date; then open questions",
    "tools_and_sources": "web search; prefer company filings and trade press; max 10 searches",
    "boundaries": "automotive only; consumer electronics is another worker's subtask; no forecasts",
}
print("vague   :", brief_problems(vague))
print("complete:", brief_problems(complete) or "OK")

> **Observe:** the "Survey 40 sources" scenario is the one place multi-agent clearly pays for itself: breadth-first, independent, context-heavy subtasks. "Debug a flaky incident" is long and exploratory but tightly coupled, so one agent loop fits better. That matches the research post's warning that multi-agent fits poorly when agents need the same context or depend on each other.

> **Try it:** add `"Migrate 300 files from library A to library B, same change each time"` to `SCENARIOS`. Which profile fits? (Claude Code's `/batch` answers it the same way: decompose into independent units, one worker each. See 8.10.)

## 8.8 Fan-out in code: orchestrator-workers with `asyncio` and the Messages API

Decompose with structured output, fan out workers with `AsyncAnthropic` under an `asyncio.Semaphore`, return failures as **structured results** instead of exceptions, and synthesize with explicit coverage gaps. Production details: the SDK retries 408/409/429/5xx and connection errors twice by default and honors `retry-after`; a shared, identical worker prefix with `cache_control` caches only after the first response starts; latency is about the slowest worker, cost is about the sum. [Theory](README.md#88-fan-out-in-code-orchestrator-workers-with-asyncio-and-the-messages-api)

In [ ]:
class WorkerBrief(BaseModel):
    id: str
    objective: str                 # what to find or produce
    boundaries: str                # what is out of scope (prevents overlap)
    output_format: str


class Decomposition(BaseModel):
    subtasks: list[WorkerBrief]
    coverage: list[CoverageItem]   # which subtask covers which aspect of the goal


class WorkerResult(BaseModel):
    task_id: str
    status: Literal["ok", "partial", "failed"]
    summary: str
    evidence: list[str]


WORKER_RULES = """You are a focused research worker inside a larger orchestrated job. Another component split the goal
into subtasks; you own exactly one of them, described in the brief you receive as JSON.

Scope
- Do only what the brief's objective asks. Respect its boundaries: anything listed as out of scope belongs to
  another worker, so do not cover it even if it seems relevant.
- If the objective is ambiguous, choose the most literal reading and say so in the summary.

Evidence
- Every claim in your summary must be supported by an item in `evidence`.
- Evidence items are short and specific: a named source, document, standard or dataset, with a date or version.
- If you rely on general knowledge rather than a specific source, write "general knowledge" as the evidence item.
  Never invent citations, URLs, statistics or quotations.
- Prefer primary sources (regulators, standards bodies, official documentation, company filings) over commentary.
- Good evidence: "EU AI Act, Regulation (EU) 2024/1689, Article 50"; "NIST AI RMF 1.0 (January 2023), Govern function".
- Bad evidence: "various sources", "industry reports", "experts say".

Status
- "ok": you answered the objective fully within the boundaries.
- "partial": you answered part of it. Say exactly which part is missing and why.
- "failed": you could not make progress. Explain what blocked you.
Returning "partial" with honest gaps is always better than a confident answer that hides them: the orchestrator
reports gaps to the reader.

Output
- Follow the brief's output_format inside `summary`, in under 200 words unless the brief says otherwise.
- `task_id` must equal the brief's id exactly.
- Write for a reader who has not seen the brief: no references to "the brief" or "my task".

Style
- Plain, specific language. Numbers with units. Dates as YYYY-MM-DD.
- No marketing language, no hedging filler, no repetition of the objective.
- If two reputable sources disagree, report both and say which you find more credible and why.

Before you answer
- Re-read the objective and the boundaries, and check that every bullet in your summary serves the objective.
- Check that every number has a unit and a source, and that every date is plausible for the claim it supports.
- If you found nothing useful, say so plainly with status "failed" rather than padding the summary.
- Remove anything another worker owns according to the boundaries; overlap wastes the reader's time and the
  orchestrator's budget."""

WORKER_SYSTEM = [{                 # identical prefix for every worker -> cacheable once above the minimum length
    "type": "text", "text": WORKER_RULES, "cache_control": {"type": "ephemeral"},
}]


async def run_worker(brief: WorkerBrief, sem: asyncio.Semaphore, llm=aclient) -> WorkerResult:
    async with sem:
        try:
            response = await llm.messages.parse(
                model=MODEL, max_tokens=1024, system=WORKER_SYSTEM,
                messages=[{"role": "user", "content": brief.model_dump_json()}],
                output_format=WorkerResult,
            )
            return parsed_or_raise(response)
        except (anthropic.APIError, ValueError, RuntimeError) as exc:   # API errors after the SDK's own retries,
            # pydantic validation errors (a ValueError), and non-end_turn stops from parsed_or_raise
            return WorkerResult(task_id=brief.id, status="failed",
                                summary=f"{type(exc).__name__}: {exc}"[:200], evidence=[])


async def fan_out(briefs: list[WorkerBrief], llm=aclient, concurrency: int = 4, warm_first: bool = False) -> list[WorkerResult]:
    sem = asyncio.Semaphore(concurrency)
    if warm_first and briefs:                       # let the first response begin, so the rest can read the cache
        first = await run_worker(briefs[0], sem, llm)
        return [first, *await asyncio.gather(*(run_worker(b, sem, llm) for b in briefs[1:]))]
    return list(await asyncio.gather(*(run_worker(b, sem, llm) for b in briefs)))


BRIEFS = [WorkerBrief(id=f"w{i:02d}", objective=f"Subtask {i}", boundaries="only this subtask",
                      output_format="3 bullets") for i in range(1, 13)]


def worker_ok(body: dict) -> dict:
    brief = json.loads(user_text(body))
    return reply_json(WorkerResult(task_id=brief["id"], status="ok", summary="- a\n- b\n- c", evidence=["general knowledge"]))


print(f"worker prefix: {len(WORKER_RULES)} chars (~{len(WORKER_RULES) // 4} tokens by a rough 4-chars-per-token estimate; "
      "check the real number with client.messages.count_tokens)")

### (a) Bounded vs unbounded concurrency (offline)

In [ ]:
for label, limit in [("unbounded (semaphore = N)", len(BRIEFS)), ("bounded (semaphore = 4)", 4)]:
    fake = FakeClaude(worker_ok, latency=0.05)
    t0 = time.perf_counter()
    results = await fan_out(BRIEFS, llm=fake.aclient, concurrency=limit)
    print(f"{label:27} peak in flight={fake.max_in_flight:2}  elapsed={time.perf_counter() - t0:.2f}s  "
          f"ok={sum(r.status == 'ok' for r in results)}/{len(results)}")

> **Observe:** the bounded run is slower in this toy (12 × 0.05 s spread over 4 slots) but never sends more than 4 requests at once. Against the real API, an unbounded burst over hundreds of tasks trips the per-model RPM / ITPM / OTPM token buckets, and the retries then make the whole run slower than a bounded one.

### (b) A 429 with `retry-after`: the SDK already backs off (offline)

In [ ]:
def worker_with_one_429() -> Callable[[dict], dict]:
    throttled: set[str] = set()

    def script(body: dict) -> dict:
        brief_id = json.loads(user_text(body))["id"]
        if brief_id == "w03" and brief_id not in throttled:
            throttled.add(brief_id)
            return api_error(429, "rate_limit_error", "Number of requests has exceeded your rate limit",
                             {"retry-after-ms": "50"})       # the real API sends `retry-after` (seconds)
        return worker_ok(body)
    return script


fake = FakeClaude(worker_with_one_429(), latency=0.01)
results = await fan_out(BRIEFS, llm=fake.aclient, concurrency=4)
print(f"default retries : {len(fake.requests)} requests for {len(BRIEFS)} briefs, statuses={Counter(r.status for r in results)}")

fake = FakeClaude(worker_with_one_429(), latency=0.01)
results = await fan_out(BRIEFS, llm=fake.aclient.with_options(max_retries=0), concurrency=4)
w03 = next(r for r in results if r.task_id == "w03")
print(f"max_retries=0   : {len(fake.requests)} requests, w03 -> {w03.status}: {w03.summary[:60]}")

### (c) Partial failures become structured results (offline)

`w05` gets a 400 (not retried), `w06` is cut off by `max_tokens` mid-JSON, `w07` honestly reports `partial`. The run still completes, and the gaps are explicit.

In [ ]:
def worker_mixed(body: dict) -> dict:
    brief_id = json.loads(user_text(body))["id"]
    if brief_id == "w05":
        return api_error(400, "invalid_request_error", "messages.0.content: brief too large")
    if brief_id == "w06":
        return reply('{"task_id": "w06", "status": "ok", "summary": "The first half of', stop_reason="max_tokens")
    if brief_id == "w07":
        return reply_json(WorkerResult(task_id="w07", status="partial", evidence=["general knowledge"],
                                       summary="Covered EU rules; US state laws not found within the search budget."))
    return worker_ok(body)


fake = FakeClaude(worker_mixed, latency=0.01)
results = await fan_out(BRIEFS, llm=fake.aclient, concurrency=4)
for r in results:
    if r.status != "ok":
        print(f"{r.task_id}: {r.status:8} {r.summary[:90]}")
print("coverage gaps to state in the synthesis:", [r.task_id for r in results if r.status != "ok"])

### (d) Prompt caching in a fan-out: send one first (offline)

The docs: *"For concurrent requests, note that a cache entry only becomes available after the first response starts. If you need cache hits for parallel requests, wait for the first response before sending subsequent requests."* The fake applies the same rule.

In [ ]:
def cache_summary(fake: FakeClaude) -> str:
    u = fake.usages
    writes = sum(x["cache_creation_input_tokens"] > 0 for x in u)
    reads = sum(x["cache_read_input_tokens"] > 0 for x in u)
    dollars = sum(cost_usd(MODEL, x) for x in u)
    return f"cache writes={writes:2} reads={reads:2}  total cost=${dollars:.4f}"


for label, warm in [("all at once", False), ("one first, then the rest", True)]:
    fake = FakeClaude(worker_ok, latency=0.03)
    await fan_out(BRIEFS, llm=fake.aclient, concurrency=12, warm_first=warm)
    print(f"{label:26} {cache_summary(fake)}")

> **Observe:** fully simultaneous requests all **write** the cache (each pays the 1.25× write price), while warming with one request turns the other eleven into cheap reads, and cached reads do not count toward ITPM on current models. The cost of warming is one worker's latency. The minimum cacheable prefix is 512 tokens on Sonnet 5.5 and Haiku 5.5 (4,096 on the legacy Haiku 4.5); below that, `cache_creation_input_tokens` stays 0 on the real API.

### (e) Estimate cost before you fan out (offline)

In [ ]:
def estimate_fanout(n: int, prefix: int, brief: int, out: int, model: str = MODEL, cached: bool = True) -> float:
    """Worker cost only: first worker writes the prefix, the rest read it (if cached)."""
    first = {"input_tokens": brief, "cache_creation_input_tokens": prefix if cached else 0, "output_tokens": out}
    rest = {"input_tokens": brief + (0 if cached else prefix), "cache_read_input_tokens": prefix if cached else 0,
            "output_tokens": out}
    if not cached:
        first["input_tokens"] += prefix
    return cost_usd(model, first) + (n - 1) * cost_usd(model, rest)


for n in (10, 100, 1000):
    realtime = estimate_fanout(n, prefix=2_000, brief=400, out=600)
    print(f"{n:5} workers: ${realtime:7.2f} real time | ${estimate_fanout(n, 2_000, 400, 600, cached=False):7.2f} uncached"
          f" | ~${realtime / 2:7.2f} via Message Batches (50% off, not latency-sensitive)")

### The README's `orchestrate()` end to end (offline)

Decompose → capped fan-out → synthesis that names the gaps. The mini-project below adds a coverage check, tracing and an independent reviewer.

In [ ]:
DECOMPOSE_SYSTEM = "Decompose the goal into independent, non-overlapping subtasks that together cover it fully."


async def orchestrate(goal: str, llm=aclient, max_workers: int = 6, concurrency: int = 4) -> dict:
    plan = parsed_or_raise(await llm.messages.parse(
        model=MODEL, max_tokens=2048, output_format=Decomposition, system=DECOMPOSE_SYSTEM,
        messages=[{"role": "user", "content": goal}],
    ))
    briefs = plan.subtasks[:max_workers]                        # cap the fan-out in code
    results = await fan_out(briefs, llm=llm, concurrency=concurrency)
    gaps = [r.task_id for r in results if r.status != "ok"]
    draft = text_of(await llm.messages.create(
        model=MODEL, max_tokens=2048,
        messages=[{"role": "user", "content":
                   f"<goal>{goal}</goal>\n<results>{[r.model_dump() for r in results]}</results>\n"
                   f"<coverage_gaps>{gaps}</coverage_gaps>\nSynthesize. State coverage gaps explicitly."}],
    ))
    return {"plan": plan, "results": results, "draft": draft, "gaps": gaps}


def orchestrate_script(body: dict) -> dict:
    if body.get("system") == DECOMPOSE_SYSTEM:
        subtasks = [WorkerBrief(id=f"w{i:02d}", objective=o, boundaries="only this aspect", output_format="3 bullets")
                    for i, o in enumerate(["EU rules", "US rules", "UK rules", "China rules", "Japan rules",
                                           "India rules", "Brazil rules", "Canada rules"], 1)]
        return reply_json(Decomposition(subtasks=subtasks, coverage=[]))
    if "<coverage_gaps>" in user_text(body):
        gaps = re.search(r"<coverage_gaps>(.*?)</coverage_gaps>", user_text(body)).group(1)
        return reply(f"Draft brief ... Coverage gaps: {gaps}.")
    return worker_mixed(body)


fake = FakeClaude(orchestrate_script, latency=0.01)
out = await orchestrate("Compare AI transparency rules across major jurisdictions", llm=fake.aclient)
print(f"planned {len(out['plan'].subtasks)} subtasks, ran {len(out['results'])} (cap 6); gaps={out['gaps']}")
print("draft:", out["draft"])

## 8.9 Decomposing multi-concern requests

Two symptoms, two fixes. If a multi-issue message makes the agent **mix up parameters**, add few-shot examples of multi-issue handling. If it makes the agent **repeat work** (far more calls than one issue needs, the same account fetched again for each issue), decompose: split the concerns, fetch shared context **once**, investigate in parallel, synthesize **one** response. [Theory](README.md#89-decomposing-multi-concern-requests)

In [ ]:
CALLS: Counter = Counter()


async def get_account(account_id: str) -> dict:
    CALLS["get_account"] += 1
    await asyncio.sleep(0.05)
    return {"id": account_id, "plan": "Unlimited 40", "monthly_fee": 45.0,
            "add_ons": {"ROAM-EU": {"paid": True, "active": False}}}


async def list_bill_lines(account: dict, month: str) -> list[dict]:
    CALLS["list_bill_lines"] += 1
    await asyncio.sleep(0.05)
    return [{"item": account["plan"], "amount": account["monthly_fee"], "month": month},
            {"item": "ROAM-EU pack", "amount": 10.0, "month": month}]


async def check_add_on(account: dict, code: str) -> dict:
    CALLS["check_add_on"] += 1
    await asyncio.sleep(0.05)
    return {"code": code, **account["add_ons"][code]}


async def plan_options(account: dict) -> dict:
    CALLS["plan_options"] += 1
    await asyncio.sleep(0.05)
    return {"current": account["plan"], "cheaper": "Essentials 15", "monthly_fee": 30.0,
            "change_effective": "start of the next billing cycle"}


class Concern(BaseModel):
    kind: Literal["bill_increase", "add_on_inactive", "plan_change", "other"]
    details: str


class Concerns(BaseModel):
    concerns: list[Concern]


MESSAGE = "My October bill went up by $10, the ROAM-EU pack I paid for still isn't active, and I want to move to a cheaper plan."
SPLIT_SYSTEM = "Split the customer's message into separate concerns. One item per concern; use 'other' if none fits."


async def investigate(concern: Concern, account: dict) -> dict:
    if concern.kind == "bill_increase":
        lines = await list_bill_lines(account, "2026-10")
        extra = [l for l in lines if l["item"] != account["plan"]]
        return {"concern": concern.kind,
                "finding": f"+${sum(l['amount'] for l in extra):.2f} is the {extra[0]['item']} charge (see the add-on concern)"}
    if concern.kind == "add_on_inactive":
        a = await check_add_on(account, "ROAM-EU")
        return {"concern": concern.kind, "finding": f"ROAM-EU paid={a['paid']} active={a['active']}: activate it and credit the missed days"}
    if concern.kind == "plan_change":
        o = await plan_options(account)
        return {"concern": concern.kind,
                "finding": f"{o['cheaper']} at ${o['monthly_fee']:.2f}/month, effective {o['change_effective']}"}
    return {"concern": concern.kind, "finding": "route to a human"}


async def naive_sequential(account_id: str, concerns: list[Concern]) -> list[dict]:
    out = []
    for c in concerns:                                  # one issue after another, refetching shared context
        account = await get_account(account_id)
        out.append(await investigate(c, account))
    return out


async def decomposed(account_id: str, concerns: list[Concern]) -> list[dict]:
    account = await get_account(account_id)             # shared context, fetched ONCE
    return list(await asyncio.gather(*(investigate(c, account) for c in concerns)))   # in parallel


def synthesize(results: list[dict]) -> str:
    return "Here is what I found and did:\n" + "\n".join(f"- {r['concern']}: {r['finding']}" for r in results)


split = FakeClaude([reply_json(Concerns(concerns=[
    Concern(kind="bill_increase", details="October bill is $10 higher"),
    Concern(kind="add_on_inactive", details="paid ROAM-EU pack is not active"),
    Concern(kind="plan_change", details="wants a cheaper plan")]))])
concerns = parsed_or_raise(split.client.messages.parse(model=MODEL, max_tokens=512, system=SPLIT_SYSTEM, output_format=Concerns,
                                                      messages=[{"role": "user", "content": MESSAGE}])).concerns

for name, strategy in [("naive sequential", naive_sequential), ("decomposed", decomposed)]:
    CALLS.clear()
    t0 = time.perf_counter()
    results = await strategy("acct_481", concerns)
    print(f"{name:17} tool calls={sum(CALLS.values())} {dict(CALLS)}  elapsed={time.perf_counter() - t0:.2f}s")
print()
print(synthesize(results))

> **Observe:** decomposition removed two redundant `get_account` calls and cut latency from the sum of all investigations to roughly the slowest one. The final answer is **one** message covering all three concerns, which is what the official CCAR-F preparation exercise asks you to verify.

> **Try it:** add a fourth concern (`"and why is my bill in USD?"`) to the scripted split. The decomposed version's latency barely changes; the sequential one grows by a full round trip.

### The other symptom: mixed-up parameters → few-shot (offline)

When the agent handles each concern fine on its own but crosses parameters between them (refunding order A for a complaint about order B), the fix is guidance inside the agent: an example of a multi-issue request handled correctly, with each issue's parameters kept separate. Splitting the message is fine (the cell above does it); handling each part in its own separate conversation is what loses the shared context (one customer, one account).

In [ ]:
MULTI_ISSUE_EXAMPLE = """<example>
Customer: "Order A-1001 arrived damaged and order A-1002 never arrived."
Reasoning: two concerns, two different orders. Keep each order's ID with its own concern.
1. get_customer(email) once.
2. lookup_order("A-1001") -> delivered, damaged -> refund A-1001 (reason: damaged).
3. lookup_order("A-1002") -> in transit, late -> open a carrier trace for A-1002. Do NOT refund A-1001 for this.
Reply: one message that addresses both orders separately.
</example>"""
SUPPORT_SYSTEM = ("You are a support agent. Customers often raise several issues at once. Handle each issue "
                  "with its own parameters, then answer in one message.\n\n" + MULTI_ISSUE_EXAMPLE)


def crossed_parameters(calls: list[dict], concern_orders: dict[str, str]) -> list[str]:
    """Flag tool calls whose order_id belongs to a different concern than the one they were made for."""
    return [f"{c['name']}({c['order_id']}) was made for concern '{c['concern']}', whose order is {concern_orders[c['concern']]}"
            for c in calls if c["order_id"] != concern_orders[c["concern"]]]


concern_orders = {"damaged": "A-1001", "late": "A-1002"}
bad_trace = [{"name": "refund", "order_id": "A-1001", "concern": "damaged"},
             {"name": "refund", "order_id": "A-1001", "concern": "late"}]
print("crossed:", crossed_parameters(bad_trace, concern_orders))
print(f"system prompt with example: {len(SUPPORT_SYSTEM)} chars")

In [ ]:
# (live) let the real model split the message
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    live = parsed_or_raise(client.messages.parse(model=MODEL, max_tokens=512, system=SPLIT_SYSTEM, output_format=Concerns,
                                                 messages=[{"role": "user", "content": MESSAGE}]))
    for c in live.concerns:
        print(f"- {c.kind}: {c.details}")

## 8.10 Decomposition in Claude Code today *(beyond the guide)*

Claude Code packages these patterns as features. You do not need their flags for CCAR-F; just know which pattern each one implements. [Theory](README.md#810-decomposition-in-claude-code-today-beyond-the-guide)

### Matching exercise (offline)

Match each of the README's four features to the pattern it implements. The checker compares hashes, so the answers are not visible in the code.

| Letter | Pattern |
|---|---|
| A | A lead coordinates **peer sessions** through a shared task list |
| B | Split a large change into **5–30 independent units**, one background worker per unit in its own git worktree (orchestrator-workers) |
| C | Several agents look for different issue classes in parallel, then a **verification step** filters false positives |
| D | A review that runs in a **background subagent with its own context** |

In [ ]:
my_matches = {
    "batch": None,         # /batch <instruction>
    "code_review": None,   # Code Review (managed)
    "local_review": None,  # /code-review (local)
    "agent_teams": None,   # Agent teams (experimental)
}

_HASHES_810 = {'batch': 'ea0122472a46789a1c37d57114a14d3de8f183aefcff35e8fef97f2303ace4f4', 'code_review': 'f16bc4d28181ccd06f60fde62bee088019cee53d3b172562d29f81f7ff682598', 'local_review': 'a47fc97a733080a7e4e33b9dd8202aa47225e22c8c6f837397ca9184942d5473', 'agent_teams': '824f2a67ac5e5b3005234a792402c9401924c4f859c499b76673c8fa6645a193'}


def check_matches(m: dict) -> None:
    score = 0
    for k, h in _HASHES_810.items():
        letter = (m.get(k) or "").strip().upper()
        ok = hashlib.sha256(f"{k}:{letter}".encode()).hexdigest() == h
        score += ok
        print(f"{'✓' if ok else '✗'} {k:14} {letter or '(blank)'}")
    print(f"{score}/{len(_HASHES_810)}")


check_matches(my_matches)

<details><summary>Answers</summary>

batch → B, code_review → C, local_review → D, agent_teams → A.
</details>

## 8.11 Decomposition for everyday Claude use (Associate lens)

In the Claude apps the same idea is a staged conversation: **clarify, gather, analyze, synthesize and draft, then review independently**, checking each stage before moving on. It is prompt chaining by hand, with you as the gate. [Theory](README.md#811-decomposition-for-everyday-claude-use-associate-lens)

In [ ]:
STAGES = [
    ("clarify", "I need a market-entry recommendation for Brazil for our B2B invoicing SaaS. Before doing anything, "
                "ask me up to 5 questions whose answers would change your recommendation."),
    ("gather", "Using my answers and the attached documents, list the facts you need for each dimension (market size, "
               "regulation, competitors). For each fact, name the document that supports it or mark it as a gap."),
    ("analyze", "Analyze each dimension separately: what the facts imply, how confident you are, what is still unknown."),
    ("draft", "Combine the three analyses into a one-page recommendation for our executive team: decision, rationale, "
              "risks, next steps."),
]
REVIEW_STAGE = ("Here is a draft recommendation and the source documents. Check every factual claim against the sources. "
                "List unsupported or contradicted claims. You did not write this draft.")


def run_staged(llm, stages, approve: Callable[[str, str], tuple[bool, str]]) -> dict:
    """One conversation for the working stages, a human gate after each, and a FRESH conversation for review."""
    messages, log = [], []
    for name, prompt in stages:
        messages.append({"role": "user", "content": prompt})
        for attempt in (1, 2):
            out = text_of(llm.messages.create(model=MODEL, max_tokens=1024, messages=messages))
            ok, feedback = approve(name, out)
            log.append(f"{name} (attempt {attempt}): {'approved' if ok else 'sent back: ' + feedback}")
            if ok:
                messages.append({"role": "assistant", "content": out})
                break
            messages += [{"role": "assistant", "content": out}, {"role": "user", "content": feedback}]
    draft = messages[-1]["content"]
    review = text_of(llm.messages.create(model=MODEL, max_tokens=1024, messages=[   # fresh: no working history
        {"role": "user", "content": f"{REVIEW_STAGE}\n\n<draft>\n{draft}\n</draft>"}]))
    return {"log": log, "draft": draft, "review": review}


staged = FakeClaude([reply("1) Target segment? 2) Budget? ..."), reply("Market size: doc 2. Regulation: GAP (no source)."),
                     reply("Analysis ... regulation unknown."), reply("Analysis ... regulation: NF-e e-invoicing mandate (doc 4)."),
                     reply("Recommendation: enter via partner ..."), reply("Unsupported claims: none found; verify doc 4 date.")])
human_gate = lambda stage, out: (False, "Regulation is still a gap: use doc 4 on e-invoicing.") if "unknown" in out else (True, "")
result = run_staged(staged.client, STAGES, human_gate)
print("\n".join(result["log"]))
print("review request had", len(staged.requests[-1]["messages"]), "message(s): independent of the working conversation")

> **Observe:** the regulation gap was caught at the **analyze** stage, where it started, not after the executive draft was written. The final review runs in a fresh conversation, the app equivalent of 8.4's independent instance.

> **Try it:** copy `STAGES` into a Claude Project that holds your source documents and run it by hand; then open a new chat for `REVIEW_STAGE`. That is the CCAO-F version of this whole chapter.

## Mini-project: a research orchestrator with an independent reviewer

Combine the chapter into one small, working system:

1. **Decompose** the goal with structured output (`Decomposition`), and check the **coverage map** against required aspects in code; re-plan once with specific feedback if something is missing (8.2).
2. **Fan out** workers with `AsyncAnthropic`: semaphore-bounded, SDK retries, a warmed cache prefix, failures captured as `failed` results, at most `max_workers` subtasks (8.8).
3. **Synthesize** with explicit coverage gaps: failed workers, dropped subtasks and uncovered aspects (8.8).
4. **Review independently**: a fresh request that sees only goal, criteria and draft, never worker transcripts (8.4).
5. **Trace** every call: step, model, tokens (including cache), cost, latency (README observability row).

It is tested offline on every branch first, then run live.

In [ ]:
class BriefReview(BaseModel):
    approved: bool
    issues: list[str]
    unsupported_claims: list[str]


PLAN_SYSTEM = ("Decompose the goal into 3-6 independent, non-overlapping subtasks that together cover it fully. "
               "Give each an id, objective, boundaries (what belongs to other subtasks) and output format. "
               "Return a coverage map with one entry per required aspect, naming the subtask id that covers it; "
               "use the aspect names exactly as given.")
BRIEF_REVIEWER_SYSTEM = ("You review a research brief you did not write. Check it against the goal and the criteria. "
                         "List concrete issues and any claims that are not supported. Approve only if it meets every criterion.")


class Trace:
    def __init__(self):
        self.rows: list[dict] = []

    def record(self, step: str, model: str, t0: float, response=None, status: str = "ok") -> None:
        u = response.usage.model_dump() if response is not None else {}
        self.rows.append({"step": step, "model": model, "status": status, "ms": int((time.perf_counter() - t0) * 1000),
                          "in": u.get("input_tokens") or 0, "cache_w": u.get("cache_creation_input_tokens") or 0,
                          "cache_r": u.get("cache_read_input_tokens") or 0, "out": u.get("output_tokens") or 0,
                          "usd": cost_usd(model, u) if u else 0.0})

    def report(self) -> None:
        print(f"{'step':16}{'status':8}{'in':>7}{'cache_w':>8}{'cache_r':>8}{'out':>6}{'usd':>9}{'ms':>7}")
        for r in self.rows:
            print(f"{r['step']:16}{r['status']:8}{r['in']:7}{r['cache_w']:8}{r['cache_r']:8}{r['out']:6}{r['usd']:9.4f}{r['ms']:7}")
        print(f"{'TOTAL':16}{'':8}{sum(r['in'] for r in self.rows):7}{sum(r['cache_w'] for r in self.rows):8}"
              f"{sum(r['cache_r'] for r in self.rows):8}{sum(r['out'] for r in self.rows):6}{sum(r['usd'] for r in self.rows):9.4f}")


async def traced_parse(llm, trace: Trace, step_name: str, **kwargs):
    t0 = time.perf_counter()
    response = await llm.messages.parse(**kwargs)
    trace.record(step_name, kwargs["model"], t0, response, status=response.stop_reason)
    return parsed_or_raise(response)


async def traced_worker(llm, trace: Trace, brief: WorkerBrief, sem: asyncio.Semaphore) -> WorkerResult:
    async with sem:
        t0 = time.perf_counter()
        try:
            return await traced_parse(llm, trace, f"worker {brief.id}", model=MODEL, max_tokens=1024, system=WORKER_SYSTEM,
                                      messages=[{"role": "user", "content": brief.model_dump_json()}],
                                      output_format=WorkerResult)
        except (anthropic.APIError, ValueError, RuntimeError) as exc:
            if not any(r["step"] == f"worker {brief.id}" for r in trace.rows):   # failed before a response arrived
                trace.record(f"worker {brief.id}", MODEL, t0, status="error")
            return WorkerResult(task_id=brief.id, status="failed", summary=f"{type(exc).__name__}: {exc}"[:200], evidence=[])


async def research_orchestrator(goal: str, required_aspects: list[str], criteria: str, llm=aclient,
                                max_workers: int = 5, concurrency: int = 4) -> dict:
    trace = Trace()
    ask = f"<goal>{goal}</goal>\n<required_aspects>{required_aspects}</required_aspects>"
    plan_kwargs = dict(model=MODEL, max_tokens=2048, system=PLAN_SYSTEM, output_format=Decomposition)

    # 1. decompose + coverage check in code, one re-plan with specific feedback
    plan = await traced_parse(llm, trace, "plan", messages=[{"role": "user", "content": ask}], **plan_kwargs)
    uncovered = coverage_gaps(required_aspects, plan.coverage)
    if uncovered:
        plan = await traced_parse(llm, trace, "re-plan", **plan_kwargs, messages=[{"role": "user", "content":
            f"{ask}\nA previous plan did not cover: {uncovered}. Return a complete plan."}])
        uncovered = coverage_gaps(required_aspects, plan.coverage)

    # 2. capped, bounded fan-out with a warmed cache prefix
    briefs, dropped = plan.subtasks[:max_workers], [b.id for b in plan.subtasks[max_workers:]]
    sem = asyncio.Semaphore(concurrency)
    results = [await traced_worker(llm, trace, briefs[0], sem)] if briefs else []
    results += await asyncio.gather(*(traced_worker(llm, trace, b, sem) for b in briefs[1:]))
    failed = [r.task_id for r in results if r.status != "ok"]
    gaps = {"uncovered_aspects": uncovered, "failed_or_partial": failed, "dropped_subtasks": dropped}

    # 3. synthesis that must state the gaps
    t0 = time.perf_counter()
    synth = await llm.messages.create(model=MODEL, max_tokens=2048, messages=[{"role": "user", "content":
        f"<goal>{goal}</goal>\n<results>{json.dumps([r.model_dump() for r in results])}</results>\n"
        f"<coverage_gaps>{json.dumps(gaps)}</coverage_gaps>\n"
        "Write the brief from the results only. State every coverage gap explicitly in a 'Gaps' section."}])
    trace.record("synthesize", MODEL, t0, synth, status=synth.stop_reason)
    draft = text_of(synth)

    # 4. independent review: goal + criteria + draft, nothing else
    review = await traced_parse(llm, trace, "review", model=MODEL, max_tokens=1024, system=BRIEF_REVIEWER_SYSTEM,
                                output_format=BriefReview, messages=[{"role": "user", "content":
                                    f"<goal>{goal}</goal>\n<criteria>{criteria}</criteria>\n<draft>\n{draft}\n</draft>"}])
    if not review.approved:
        status = "needs_revision"
    elif any(gaps.values()):
        status = "ready_with_gaps"
    else:
        status = "ready"
    return {"status": status, "plan": plan, "results": results, "gaps": gaps, "draft": draft, "review": review, "trace": trace}


print("orchestrator ready")

### Test every branch offline

`ProjectScript` plays every role: planner (a list of plans, served in order), workers (behavior per brief id), synthesizer (echoes the gaps it was given) and reviewer. Each test asserts on the **result** and on **what was sent** (for example, that the reviewer never sees worker output).

In [ ]:
ASPECTS = ["security", "compliance", "cost"]
CRITERIA_BRIEF = "Covers every required aspect or names it as a gap. Every claim traceable to a worker result. Under 400 words."


def make_plan(n: int, aspects: list[str]) -> Decomposition:
    subtasks = [WorkerBrief(id=f"t{i}", objective=f"Research aspect {i}", boundaries="this aspect only",
                            output_format="bullets with evidence") for i in range(1, n + 1)]
    return Decomposition(subtasks=subtasks, coverage=[CoverageItem(aspect=a, subtask_id=f"t{i}") for i, a in enumerate(aspects, 1)])


class ProjectScript:
    def __init__(self, plans: list[Decomposition], workers: dict[str, str] | None = None, approve: bool = True):
        self.plans, self.workers, self.approve = list(plans), workers or {}, approve

    def __call__(self, body: dict) -> dict:
        system = body.get("system")
        if system == PLAN_SYSTEM:
            return reply_json(self.plans.pop(0))
        if system == BRIEF_REVIEWER_SYSTEM:
            return reply_json(BriefReview(approved=self.approve, unsupported_claims=[],
                                          issues=[] if self.approve else ["'cost' section has no evidence"]))
        if isinstance(system, list):                                   # a worker (WORKER_SYSTEM)
            brief_id = json.loads(user_text(body))["id"]
            behavior = self.workers.get(brief_id, "ok")
            if behavior == "400":
                return api_error(400, "invalid_request_error", "bad brief")
            if behavior == "truncated":
                return reply('{"task_id": "' + brief_id + '", "status": "ok", "summ', stop_reason="max_tokens")
            return reply_json(WorkerResult(task_id=brief_id, status=behavior, summary=f"findings for {brief_id}",
                                           evidence=[f"source-{brief_id}"]))
        gaps = re.search(r"<coverage_gaps>(.*?)</coverage_gaps>", user_text(body)).group(1)
        return reply(f"## Brief\n...\n## Gaps\n{gaps}")                 # synthesizer


async def run_case(script: ProjectScript, **kw) -> tuple[dict, FakeClaude]:
    fake = FakeClaude(script, latency=0.01)
    out = await research_orchestrator("Assess adopting AI coding agents in a regulated bank", ASPECTS, CRITERIA_BRIEF,
                                      llm=fake.aclient, **kw)
    return out, fake


# 1. happy path
out, fake = await run_case(ProjectScript([make_plan(3, ASPECTS)]))
review_req = fake.requests[-1]
assert out["status"] == "ready" and len(fake.requests) == 1 + 3 + 1 + 1
assert "findings for t1" not in json.dumps(review_req) and WORKER_RULES[:40] not in json.dumps(review_req), "reviewer saw worker context"
assert sum(r["cache_r"] > 0 for r in out["trace"].rows) == 2, "workers 2 and 3 should read the warmed cache"
print("✓ 1 happy path: ready, 6 calls, reviewer isolated, 1 cache write + 2 reads")

# 2. coverage gap -> one re-plan
out, fake = await run_case(ProjectScript([make_plan(2, ASPECTS[:2]), make_plan(3, ASPECTS)]))
assert [r["step"] for r in out["trace"].rows][:2] == ["plan", "re-plan"] and out["gaps"]["uncovered_aspects"] == []
assert "did not cover: ['cost']" in user_text(fake.requests[1])
print("✓ 2 coverage gap: re-planned with specific feedback, gap closed")

# 3. too many subtasks -> capped, dropped ones reported
out, fake = await run_case(ProjectScript([make_plan(7, ASPECTS)]), max_workers=5)
assert out["gaps"]["dropped_subtasks"] == ["t6", "t7"] and out["status"] == "ready_with_gaps"
assert "t6" in out["draft"]
print("✓ 3 cap: ran 5 of 7, dropped t6/t7 stated in the draft")

# 4. partial failures -> run completes, gaps explicit
out, fake = await run_case(ProjectScript([make_plan(4, ASPECTS)], workers={"t2": "400", "t3": "truncated", "t4": "partial"}))
assert out["gaps"]["failed_or_partial"] == ["t2", "t3", "t4"] and "t3" in out["draft"]
assert any(r["status"] == "error" for r in out["trace"].rows)
print("✓ 4 failures: 400 + truncation + partial captured, synthesis names them")

# 5. reviewer rejects
out, fake = await run_case(ProjectScript([make_plan(3, ASPECTS)], approve=False))
assert out["status"] == "needs_revision" and out["review"].issues
print("✓ 5 reviewer rejection surfaces as needs_revision:", out["review"].issues)
print()
out["trace"].report()

> **Observe:** the trace shows where the money goes: the planner and reviewer are single calls, while worker cost scales with N (and the warmed cache makes workers 2..N cheaper on input). The status is never just "done": `ready_with_gaps` and `needs_revision` are first-class outcomes that a UI or a human reviewer can act on.

> **Try it:** add a sixth branch in which the **planner** returns a plan with zero subtasks. What should the orchestrator do? (Hint: the current code would synthesize from nothing; return a `no_plan` status before fanning out.)

In [ ]:
# (live) run the orchestrator for real (~8 calls, roughly $0.05-0.15 with the default model)
if not HAS_KEY:
    print("skipped (no ANTHROPIC_API_KEY)")
else:
    out = await research_orchestrator(
        "Write a one-page brief for a bank CTO on adopting AI coding agents for internal software development.",
        required_aspects=["security", "regulatory compliance", "code quality", "cost", "change management"],
        criteria=CRITERIA_BRIEF, max_workers=5, concurrency=3,
    )
    print("status:", out["status"], "| gaps:", out["gaps"])
    print("review:", out["review"].model_dump())
    print(out["draft"][:1500])
    out["trace"].report()

## Recap

- **Diagnose the shape first.** Known, dependent steps → a gated chain; independent slices → sectioning; distinct categories → routing; unknown subtasks → orchestrator-workers or a planner loop; long, exploratory, environment-driven work → an agent.
- **Gates are code** and decide on structured fields. Planner loops stop on a boolean, with a round cap as a safety net, and report budget exhaustion.
- **Coverage maps** catch too-narrow decompositions before the fan-out spends money.
- **Large reviews:** per-file local passes with one rubric, then an integration pass over interfaces and findings, then dedupe in code. Not a bigger context, not majority voting.
- **Independence is an input decision:** the reviewer gets artifact + spec + criteria, never the generator's conversation.
- **Verification + calibrated routing:** per-finding confidence is useful only after you measure precision per bucket; uncertain findings go to humans, not to `/dev/null`.
- **Fan-out in production:** semaphore, SDK retries with `retry-after`, warm the cache with one request first, failures as structured results, trace tokens and cost; use Message Batches when nobody is waiting.

Next: [homework](02_homework.ipynb)